In [ ]:
# Install required libraries
!pip install -q openpyxl pyarrow scikit-learn pandas numpy

import os
from urllib.parse import urlparse

import numpy as npy
import pandas as pda
from google.colab import files


In [ ]:
# 1. Upload Dataset into Google Colab

print("Upload 'cybersecurity.xlsx':")
uploaded = files.upload()
input_path = list(uploaded.keys())[0]

Upload 'cybersecurity.xlsx':


Saving cybersecurity.xlsx to cybersecurity (2).xlsx


In [ ]:
# 2. Helper Functions
def bin_port(port: int) -> str:
    """Group Layer 4 ports into standard IANA functional ranges."""
    if port <= 1023:
        return "system"
    elif port <= 49151:
        return "registered"
    return "ephemeral"


def extract_url_path(url_value: str) -> str:
    """Strip random query IDs (e.g., ?id=12345) and isolate clean endpoint paths."""
    if url_value == "No_URL":
        return "none"
    parsed = urlparse(url_value)
    return parsed.path.lower() if parsed.path else "root"

In [ ]:
# 3. Run Curation & Feature Engineering Pipeline

output_dir = "/Milestone1/curated_data"
os.makedirs(output_dir, exist_ok=True)

# Load Workbook & Verify Schema
xls = pda.ExcelFile(input_path)
print(f"Sheets found: {xls.sheet_names}")
df = pda.read_excel(input_path, sheet_name="cybersecurity")

assert df.shape == (10000, 13), f"Unexpected shape: {df.shape}"
assert df["src_port"].between(0, 65535).all(), "Invalid src_port values found."
assert df["dst_port"].between(0, 65535).all(), "Invalid dst_port values found."
assert (df["bytes_sent"] >= 0).all() and (
    df["bytes_received"] >= 0
).all(), "Negative byte counts found."
# Duplicate Audit & Missing Value Imputation
dup_count = df.duplicated().sum()
print(f"Exact duplicates found: {dup_count}")
df = df.drop_duplicates().copy()

# Impute 3,232 missing URLs and create binary indicator
df["has_url"] = df["url"].notnull().astype(int)
df["url"] = df["url"].fillna("No_URL")

# Temporal Extraction
df["timestamp"] = pda.to_datetime(df["timestamp"])
df["hour_of_day"] = df["timestamp"].dt.hour
df["is_weekend"] = (df["timestamp"].dt.dayofweek >= 5).astype(int)

# IANA Port Range Binning
df["src_port_bin"] = df["src_port"].apply(bin_port)
df["dst_port_bin"] = df["dst_port"].apply(bin_port)

# Clean URL Endpoint Extraction
df["url_path"] = df["url"].apply(extract_url_path)

# Log Transformations & Directional Byte Ratios
df["log_bytes_sent"] = npy.log1p(df["bytes_sent"])
df["log_bytes_received"] = npy.log1p(df["bytes_received"])
df["log_total_bytes"] = npy.log1p(df["bytes_sent"] + df["bytes_received"])
df["byte_ratio"] = df["log_bytes_sent"] - df["log_bytes_received"]

# Boolean to Integer
df["is_internal_traffic"] = df["is_internal_traffic"].astype(int)

# Drop 100% Unique / Leakage-Prone Identifiers (src_ip, dst_ip, raw timestamp, raw url)
drop_cols = ["timestamp", "src_ip", "dst_ip", "url"]
df_clean = df.drop(columns=drop_cols)

# One-Hot Encode Categorical Predictors
cat_cols = ["protocol", "user_agent", "src_port_bin", "dst_port_bin", "url_path"]
df_encoded = pda.get_dummies(df_clean, columns=cat_cols, drop_first=False, dtype=int)

print(f"Curated & encoded dataset shape: {df_encoded.shape}")



Sheets found: ['cybersecurity']
Exact duplicates found: 0
Curated & encoded dataset shape: (10000, 51)
